# 01 — Bronze Layer

This notebook ingests the full source files from ADLS Gen2 and refreshes the Bronze Delta tables.
It is reused by the incremental solution because the incremental logic is implemented in Silver and Gold.


In [ ]:
CATALOG = "sales_store"
SCHEMA = "linio"

STORAGE_ACCOUNT = "stsalesstoresergio"
CONTAINER = "sales-store"

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

FILE_PURCHASES_IN_PERSON = BASE_PATH + "landing/compras/Presencial.csv"
FILE_PURCHASES_ONLINE = BASE_PATH + "landing/compras/Online.json"
FILE_DETAILS = BASE_PATH + "landing/detalles/*.csv"

BRONZE_PURCHASES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_compras"
BRONZE_DETAILS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_detalles"


In [ ]:
import re

from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StructField, StructType, StringType

def normalize_column_names(df):
    normalized = [
        re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", column_name).lower()
        for column_name in df.columns
    ]
    return df.toDF(*normalized)


## Read in-person purchases


In [ ]:
purchases_in_person = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .load(FILE_PURCHASES_IN_PERSON)
)

purchases_in_person = (
    normalize_column_names(purchases_in_person)
    .withColumn("tipo_compra", lit("Presencial"))
    .withColumn("fecha_carga", current_timestamp())
)


## Read online purchases


In [ ]:
# Read once to capture the actual JSON field names.
online_raw = (
    spark.read
    .option("multiline", True)
    .json(FILE_PURCHASES_ONLINE)
)

# Force every source field to StringType, as required for Bronze.
online_schema = StructType([
    StructField(column_name, StringType(), True)
    for column_name in online_raw.columns
])

purchases_online = (
    spark.read
    .format("json")
    .schema(online_schema)
    .option("multiline", True)
    .load(FILE_PURCHASES_ONLINE)
)

purchases_online = (
    normalize_column_names(purchases_online)
    .withColumn("tipo_compra", lit("Online"))
    .withColumn("fecha_carga", current_timestamp())
)


## Combine purchase channels


In [ ]:
df_purchases = purchases_in_person.unionByName(
    purchases_online,
    allowMissingColumns=True
)


## Read detail files


In [ ]:
df_details = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "|")
    .option("mergeSchema", True)
    .load(FILE_DETAILS)
)

df_details = normalize_column_names(df_details)

# Keep the assignment-required column even when an input batch does not contain it.
if "oferta_id" not in df_details.columns:
    df_details = df_details.withColumn("oferta_id", lit(None).cast("string"))

df_details = (
    df_details
    .withColumn("nombre_archivo", col("_metadata.file_name"))
    .withColumn("fecha_carga", current_timestamp())
)


## Refresh Bronze tables


In [ ]:
(
    df_purchases.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PURCHASES_TABLE)
)

(
    df_details.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DETAILS_TABLE)
)

print("Bronze refresh completed.")
print(f"Purchases: {spark.table(BRONZE_PURCHASES_TABLE).count():,}")
print(f"Details: {spark.table(BRONZE_DETAILS_TABLE).count():,}")
